# imgjit — Colab build/test runner

Runtime must be set to **T4 GPU** (`Runtime → Change runtime type`) before running this notebook.

See `docs/PLAN.md` for which phase this is verifying and `docs/ARCHITECTURE.md` / `docs/PROTOCOL.md`
for design context. Run cells top to bottom; the clone/pull cell is safe to re-run after every
`git push` from your Mac — no need to restart the runtime.

In [ ]:
import os

REPO_URL = "https://github.com/zmx27/Image-Processing-Engine.git"
REPO_DIR = "/content/Image-Processing-Engine"

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

## Toolchain / GPU sanity check

Run once per session to confirm you actually got a GPU and to catch CUDA/driver version drift
between sessions before it shows up as a confusing build or runtime failure.

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version

## Configure + build

`find_package(CUDAToolkit)` in the root `CMakeLists.txt` should report the CUDA backend enabled
here (unlike on a Mac, which always falls back to the CPU-only portable core).

In [ ]:
%cd {REPO_DIR}
!cmake -B build -DCMAKE_BUILD_TYPE=RelWithDebInfo
!cmake --build build -j

## Run tests

In [ ]:
%cd {REPO_DIR}
!ctest --test-dir build --output-on-failure

## Phase-specific runs

Cells below are added incrementally as each phase in `docs/PLAN.md` produces something runnable
(e.g. the Phase 2/3 `imgjit-cli`, the Phase 8 `bench` harness).

### Phase 1 — driver API + JIT spike

`imgjit-spike` inverts a PNG on-GPU twice — once from PTX that `nvcc` built ahead of time
(checkpoint 1a, driver plumbing with JIT out of the picture), then once from PTX that NVRTC
compiled at runtime (checkpoint 1b) — and diffs both against a scalar CPU loop. Inversion is an
integer pointwise op, so the bar is exact equality. It exits non-zero on any mismatch, so the
`ctest` cell above already covers it; run it directly to see the per-checkpoint output and to
inspect the generated PTX.

In [ ]:
%cd {REPO_DIR}
!./build/tools/imgjit-spike tests/testdata/gradient_32x32_rgba.png \
    --out /content/inverted.png --dump-ptx /content/invert_jit.ptx

# The generated PTX is what you read when debugging codegen (Phase 3 adds --dump-source
# for the CUDA side of the same idea).
!head -30 /content/invert_jit.ptx